# Water Pipe Renewal Priority – Waimakariri (NZ)

Risk-based renewal priority of a real water supply network: **likelihood of failure** (pipe age, material, condition) × **consequence of failure** (criticality) = **risk score** and **priority class** for every network main.

**Data:** Waimakariri District Council (New Zealand) – public ArcGIS REST service `3Waters / Assets_Water_Supply / Water Supply Pipe`, loaded in QGIS and saved as GeoPackage (`waimakariri_mains.gpkg`, layer `water_mains`). Same data as project *Water Network Analysis – Waimakariri* (analysis 1).

In [ ]:
# 1. Install packages
!pip install -q contextily

In [ ]:
# 2. Imports and settings
import os, re, warnings
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
warnings.filterwarnings("ignore")
os.makedirs("images", exist_ok=True); os.makedirs("results", exist_ok=True); os.makedirs("data", exist_ok=True)
GPKG = "data/waimakariri_mains.gpkg"
BEZUGSJAHR = 2026
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
def de(x, d=1):
    return f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
FMT = FuncFormatter(lambda v, _: de(v, 0))

In [ ]:
# 3. Upload the GeoPackage (only in Google Colab, if it is not in data/ yet)
if not os.path.exists(GPKG):
    from google.colab import files
    up = files.upload()
    os.rename(list(up)[0], GPKG)
pipes = gpd.read_file(GPKG, layer="water_mains")
print(len(pipes), "pipes |", pipes.crs)

In [ ]:
# 4. Network mains + scores
# Likelihood of failure (1-5) = 40 % age + 40 % material + 20 % condition
# Consequence of failure (1-4) = criticality (AA=4, A=3, B=2, C=1, unknown=2)
# Risk = likelihood x consequence (1-20)
hl = pipes[pipes["CLASSIFICATION3"] == "Network Main"].copy()
hl["laenge_km"] = hl.geometry.length / 1000           # EPSG:2193, metres
hl["baujahr"] = hl["ACQN_DATE"].dt.year              # install year derived from ACQN_DATE
hl["gebiet"] = hl["SCHEME_NAME"].fillna("unbekannt").str.replace("W - ", "", regex=False).str.title()

alter = BEZUGSJAHR - hl["baujahr"]
hl["p_alter"] = np.select([alter < 20, alter < 40, alter < 60, alter < 80, alter >= 80], [1, 2, 3, 4, 5], 3)

def p_werkstoff(m):
    m = "" if pd.isna(m) else m
    if m == "ASBESTOS CEMENT": return 5                       # asbestos cement
    if "STEEL" in m.upper() or "IRON" in m.upper(): return 4  # steel / cast iron
    if m in ["ALKATHENE", "PE - LOW DENSITY"]: return 3       # early PE
    if "PVC" in m.upper(): return 2
    if m.startswith("PE") or m.startswith("Poly"): return 1   # modern PE
    return 3                                                  # other / unknown
hl["p_werkstoff"] = hl["MATERIAL_DISPLAY"].map(p_werkstoff)
zustand = pd.to_numeric(hl["ASSET_CONDITION"], errors="coerce")
hl["p_zustand"] = zustand.where(zustand.between(1, 5), 3)    # unknown condition = 3 (neutral)
hl["wahrscheinlichkeit"] = (0.4 * hl["p_alter"] + 0.4 * hl["p_werkstoff"] + 0.2 * hl["p_zustand"]).round(1)

krit = hl["CRITICALITY_RATING"].fillna("")
hl["folge"] = np.select([krit.str.startswith("AA"), krit.str.startswith("A"), krit == "B", krit == "C"], [4, 3, 2, 1], 2)
hl["risiko"] = (hl["wahrscheinlichkeit"] * hl["folge"]).round(1)

PRIO = ["Prio 1 – sehr hoch", "Prio 2 – hoch", "Prio 3 – mittel", "Prio 4 – gering"]
PRIO_FARBEN = dict(zip(PRIO, ["#B2182B", "#F4A582", "#92C5DE", "#D9D9D9"]))
hl["prioritaet"] = np.select([hl["risiko"] >= 10, hl["risiko"] >= 7, hl["risiko"] >= 4], PRIO[:3], PRIO[3])

def strasse(s):
    s = "" if pd.isna(s) else str(s)
    s = re.sub(r"(?i)^.*?pressure\s*:?\s*", "", s)
    s = re.sub(r"^.*:\s*", "", s)                 # e.g. 'Rising Main: West Belt'
    s = re.sub(r"^[-\s\d]+", "", s)               # e.g. '- 1 - 19 Gerard Close'
    return s.strip().title() or "unbekannt"
hl["strasse"] = hl["LONG_DESCRIPTION"].map(strasse)
print(f"{len(hl):,} network mains, {hl['laenge_km'].sum():,.1f} km")
hl.groupby("prioritaet")["laenge_km"].sum().round(1)

In [ ]:
# 5. Tables and key figures
gesamt = hl["laenge_km"].sum()
t_prio = hl.groupby("prioritaet")["laenge_km"].agg(["count", "sum"]).reindex(PRIO)
t_prio.columns = ["Abschnitte", "Länge (km)"]
t_prio["Anteil (%)"] = 100 * t_prio["Länge (km)"] / gesamt
t_prio.index.name = "Priorität"; t_prio = t_prio.round(2)

def ws_gruppe(m):
    return {5: "Asbestzement (AZ)", 4: "Stahl / Guss", 2: "PVC", 1: "PE"}.get(p_werkstoff(m), "PE alt / Sonstige / unbekannt")
hl["werkstoff"] = hl["MATERIAL_DISPLAY"].map(ws_gruppe)
WS_REIHE = ["Asbestzement (AZ)", "Stahl / Guss", "PVC", "PE", "PE alt / Sonstige / unbekannt"]
ws_prio = hl.pivot_table(index="werkstoff", columns="prioritaet", values="laenge_km", aggfunc="sum", fill_value=0)
ws_prio = ws_prio.reindex(index=[w for w in WS_REIHE if w in ws_prio.index], columns=PRIO, fill_value=0).round(2)
geb_prio = hl.pivot_table(index="gebiet", columns="prioritaet", values="laenge_km", aggfunc="sum", fill_value=0)
geb_prio = geb_prio.reindex(columns=PRIO, fill_value=0)
geb_prio = geb_prio.sort_values(PRIO[:2], ascending=False).round(2)
matrix = hl.assign(w=hl["wahrscheinlichkeit"].round().astype(int)).pivot_table(
    index="w", columns="folge", values="laenge_km", aggfunc="sum", fill_value=0).reindex(index=range(5, 0, -1), columns=range(1, 5), fill_value=0)
matrix.index.name = "Wahrscheinlichkeit"; matrix.columns.name = "Folge"; matrix = matrix.round(2)

spalten = {"ASSET_ID": "Asset-ID", "strasse": "Straße", "gebiet": "Versorgungsgebiet", "MATERIAL_DISPLAY": "Werkstoff",
           "DIAMETER_mm": "DN (mm)", "baujahr": "Baujahr", "ASSET_CONDITION_DISPLAY": "Zustand",
           "CRITICALITY_RATING": "Kritikalität", "wahrscheinlichkeit": "Wahrscheinlichkeit (1–5)",
           "folge": "Folge (1–4)", "risiko": "Risiko (1–20)", "prioritaet": "Priorität"}
liste = hl.sort_values(["risiko", "laenge_km"], ascending=False)
liste = liste.assign(**{"Länge (m)": (liste["laenge_km"] * 1000).round(1)})
liste = liste[list(spalten) + ["Länge (m)"]].rename(columns=spalten)
top_abschnitte = liste.head(30)
prio12 = liste[liste["Priorität"].isin(PRIO[:2])]

top_str = (hl[hl["prioritaet"].isin(PRIO[:2])].groupby(["strasse", "gebiet"])
           .apply(lambda d: pd.Series({"Länge Prio 1+2 (km)": d["laenge_km"].sum(),
                                       "davon Prio 1 (km)": d.loc[d["prioritaet"] == PRIO[0], "laenge_km"].sum(),
                                       "max. Risiko": d["risiko"].max(),
                                       "Hauptwerkstoff": d.groupby("werkstoff")["laenge_km"].sum().idxmax()}))
           .reset_index().rename(columns={"strasse": "Straße", "gebiet": "Versorgungsgebiet"}))
top_str = top_str[top_str["Straße"] != "unbekannt"].sort_values(["davon Prio 1 (km)", "Länge Prio 1+2 (km)"], ascending=False)
top_str = top_str.head(20).round(2).reset_index(drop=True); top_str.index += 1

p1 = hl[hl["prioritaet"] == PRIO[0]]; p2 = hl[hl["prioritaet"] == PRIO[1]]
kz = pd.DataFrame({"Kennzahl": [
    "Hauptleitungen (Abschnitte)", "Hauptleitungen (km)", "Prio 1 – sehr hoch (km)", "Prio 2 – hoch (km)",
    "Prio 1+2 (Anteil % der Netzlänge)", "Prio 1: Anteil Asbestzement (%)", "Prio 1+2: Anteil Asbestzement (%)",
    "Prio 1: mittleres Baujahr (längengewichtet)", "Prio 1+2: Länge mit Zustandsnote (%)",
    "Maximales Risiko", "Abschnitte mit Risiko ≥ 10"],
  "Wert": [len(hl), round(gesamt, 1), round(p1["laenge_km"].sum(), 1), round(p2["laenge_km"].sum(), 1),
    round(100 * (p1["laenge_km"].sum() + p2["laenge_km"].sum()) / gesamt, 1),
    round(100 * p1.loc[p1["werkstoff"] == "Asbestzement (AZ)", "laenge_km"].sum() / p1["laenge_km"].sum(), 1),
    round(100 * pd.concat([p1, p2]).query("werkstoff == 'Asbestzement (AZ)'")["laenge_km"].sum()
          / (p1["laenge_km"].sum() + p2["laenge_km"].sum()), 1),
    round((p1["baujahr"] * p1["laenge_km"]).sum() / p1.loc[p1["baujahr"].notna(), "laenge_km"].sum(), 0),
    round(100 * pd.concat([p1, p2]).loc[lambda d: pd.to_numeric(d["ASSET_CONDITION"], errors="coerce").between(1, 5), "laenge_km"].sum()
          / (p1["laenge_km"].sum() + p2["laenge_km"].sum()), 1),
    hl["risiko"].max(), int((hl["risiko"] >= 10).sum())]})
methode = pd.DataFrame({"Faktor": ["Alter", "Alter", "Alter", "Alter", "Alter", "Werkstoff", "Werkstoff", "Werkstoff", "Werkstoff", "Werkstoff",
                                   "Zustand", "Zustand", "Folge (Kritikalität)", "Folge (Kritikalität)", "Folge (Kritikalität)", "Folge (Kritikalität)", "Folge (Kritikalität)"],
    "Klasse": ["< 20 Jahre", "20–39 Jahre", "40–59 Jahre", "60–79 Jahre", "≥ 80 Jahre",
               "Asbestzement", "Stahl / Guss", "PE alt (Alkathene, PE-LD), Sonstige, unbekannt", "PVC", "PE (MD, HD, PE80, PE100)",
               "Note 1–5", "keine Note", "AA", "A", "B", "C", "unbekannt"],
    "Punkte": [1, 2, 3, 4, 5, 5, 4, 3, 2, 1, "1–5", 3, 4, 3, 2, 1, 2]})

t_prio.to_csv("results/laenge_je_prioritaet.csv"); ws_prio.to_csv("results/werkstoff_x_prioritaet.csv")
geb_prio.to_csv("results/versorgungsgebiet_x_prioritaet.csv"); top_str.to_csv("results/top20_strassen.csv")
prio12.to_csv("results/prioritaetenliste_prio1_2.csv", index=False); kz.to_csv("results/kennzahlen.csv", index=False)
with pd.ExcelWriter("results/erneuerungsprioritaet_waimakariri.xlsx") as xw:
    kz.to_excel(xw, sheet_name="Kennzahlen", index=False); methode.to_excel(xw, sheet_name="Methode", index=False)
    t_prio.to_excel(xw, sheet_name="Prioritaet"); top_str.to_excel(xw, sheet_name="Top 20 Strassen")
    prio12.to_excel(xw, sheet_name="Prioritaetenliste Prio 1+2", index=False)
    ws_prio.to_excel(xw, sheet_name="Werkstoff x Prioritaet"); geb_prio.to_excel(xw, sheet_name="Gebiet x Prioritaet")
    matrix.to_excel(xw, sheet_name="Risikomatrix (km)")
hl[["ASSET_ID", "strasse", "gebiet", "MATERIAL_DISPLAY", "DIAMETER_mm", "baujahr", "ASSET_CONDITION_DISPLAY", "CRITICALITY_RATING",
    "p_alter", "p_werkstoff", "p_zustand", "wahrscheinlichkeit", "folge", "risiko", "prioritaet", "geometry"]].to_file(
    "results/erneuerungsprioritaet_waimakariri.gpkg", layer="prioritaet", driver="GPKG")
kz

In [ ]:
# 6. Maps (Esri Light Gray basemap)
def grundkarte(ax, daten):
    try:
        import contextily as ctx
        ctx.add_basemap(ax, crs=daten.crs, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=6)
    except Exception as e:
        print("Basemap not available:", e)

def karte(daten, titel, datei, orte, bbox=None):
    km = bbox is None                                  # km totals only on the full map
    fig, ax = plt.subplots(figsize=(12, 6) if bbox is None else (10, 8))
    for p, lw in zip(PRIO[::-1], [0.5, 1.0, 1.8, 2.8]):
        sub = daten[daten["prioritaet"] == p]
        if len(sub):
            sub.plot(ax=ax, color=PRIO_FARBEN[p], linewidth=lw, label=f"{p} ({de(sub['laenge_km'].sum(), 1)} km)" if km else p, zorder=PRIO[::-1].index(p) + 2)
    if bbox is not None:
        ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])
    grundkarte(ax, daten)
    for ort in orte:
        c = hl.loc[hl["gebiet"] == ort].geometry.union_all().centroid
        ax.annotate(ort.replace(" No 1", ""), (c.x, c.y), xytext=(0, 14), textcoords="offset points", ha="center",
                    fontsize=9, fontweight="bold", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8), zorder=10)
    ax.set_axis_off()
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[::-1], l[::-1], loc="lower left", fontsize=9, title="Erneuerungspriorität", title_fontsize=9, framealpha=0.9)
    ax.set_title(titel, fontsize=13)
    fig.savefig(f"images/{datei}", dpi=200, bbox_inches="tight"); plt.show()

karte(hl, "Erneuerungspriorität – Hauptleitungen Waimakariri (Risiko = Wahrscheinlichkeit × Folge)",
      "01_Erneuerungsprioritaet_Waimakariri.png", ["Rangiora", "Kaiapoi", "Woodend", "Oxford No 1", "Mandeville"])
c = hl.loc[hl["gebiet"] == "Rangiora"].geometry.centroid          # town centre: 10-90 % of the pipe centroids
x0, x1, y0, y1 = c.x.quantile(0.1), c.x.quantile(0.9), c.y.quantile(0.1), c.y.quantile(0.9)
karte(hl, "Erneuerungspriorität – Rangiora", "02_Prioritaet_Rangiora.png", [],
      bbox=(x0 - 300, y0 - 300, x1 + 300, y1 + 300))

In [ ]:
# 7. Charts
fig, ax = plt.subplots(figsize=(6.5, 4.6))
werte = matrix.values
ax.imshow([[w * f for f in range(1, 5)] for w in range(5, 0, -1)], cmap="RdYlBu_r", vmin=1, vmax=20, alpha=0.75)
for i in range(5):
    for j in range(4):
        ax.text(j, i, f"{de(werte[i, j], 1)} km", ha="center", va="center", fontsize=9)
ax.set_xticks(range(4), ["1 (C)", "2 (B)", "3 (A)", "4 (AA)"]); ax.set_yticks(range(5), [5, 4, 3, 2, 1])
ax.set_xlabel("Folge (Kritikalität)"); ax.set_ylabel("Ausfallwahrscheinlichkeit")
ax.set_title("Risikomatrix – Länge der Hauptleitungen"); ax.spines[:].set_visible(False)
fig.tight_layout(); fig.savefig("images/03_Risikomatrix.png", dpi=200); plt.show()

def gestapelt(tab, titel, datei, h=4):
    fig, ax = plt.subplots(figsize=(8, h)); links = np.zeros(len(tab))
    for p in PRIO:
        ax.barh(tab.index[::-1], tab[p][::-1], left=links, color=PRIO_FARBEN[p], label=p); links += tab[p][::-1].values
    ax.xaxis.set_major_formatter(FMT); ax.set_xlabel("Länge (km)"); ax.legend(frameon=False, fontsize=8)
    ax.set_title(titel); fig.tight_layout(); fig.savefig(f"images/{datei}", dpi=200); plt.show()
gestapelt(ws_prio, "Priorität nach Werkstoff", "04_Prioritaet_Werkstoff.png", 3.6)

g12 = geb_prio[geb_prio[PRIO[0]] + geb_prio[PRIO[1]] > 0][PRIO[:2]]
fig, ax = plt.subplots(figsize=(8, 4)); links = np.zeros(len(g12))
for p in PRIO[:2]:
    ax.barh(g12.index[::-1], g12[p][::-1], left=links, color=PRIO_FARBEN[p], label=p); links += g12[p][::-1].values
for y, v in enumerate(links):
    ax.text(v + 0.2, y, de(v, 1), va="center", fontsize=8)
ax.set_xlabel("Länge (km)"); ax.legend(frameon=False, fontsize=8); ax.set_title("Prio 1 und 2 nach Versorgungsgebiet")
fig.tight_layout(); fig.savefig("images/05_Prioritaet_Versorgungsgebiet.png", dpi=200); plt.show()

t10 = top_str.head(10)
fig, ax = plt.subplots(figsize=(8, 4))
lab = (t10["Straße"] + " (" + t10["Versorgungsgebiet"].str.replace(" No 1", "").str.replace(" No 2", " 2") + ")")[::-1]
ax.barh(lab, t10["davon Prio 1 (km)"][::-1], color=PRIO_FARBEN[PRIO[0]], label=PRIO[0])
ax.barh(lab, (t10["Länge Prio 1+2 (km)"] - t10["davon Prio 1 (km)"])[::-1], left=t10["davon Prio 1 (km)"][::-1],
        color=PRIO_FARBEN[PRIO[1]], label=PRIO[1])
ax.set_xlabel("Länge (km)"); ax.legend(frameon=False, fontsize=8); ax.set_title("Top 10 Straßen für die Erneuerung")
fig.tight_layout(); fig.savefig("images/06_Top10_Strassen.png", dpi=200); plt.show()

In [ ]:
# 8. Download images and results (Google Colab)
!zip -qr renewal_priority_results.zip images results
try:
    from google.colab import files
    files.download("renewal_priority_results.zip")
except ImportError:
    pass